# MORICHIKA Gemma 4 31B — full-corpus v5

Clean native 31B QAT Q4_0 inference; 12 hash-bound searchable source families; strict typed contextual grammar lookup; no legacy notebook/data dump; writes but never submits submission.csv.

In [1]:
from __future__ import annotations

import gc, hashlib, importlib, importlib.metadata, json, math, os, random, re
import shutil, sqlite3, stat, subprocess, sys, time, unicodedata, zipfile
from collections import Counter, defaultdict
from pathlib import Path, PurePosixPath
from typing import Any, Callable

TOTAL_RUN_STARTED_MONOTONIC = time.monotonic()
TOTAL_DEADLINE_SECONDS = 8 * 60 * 60 + 15 * 60

import numpy as np
import pandas as pd

SEED = 20260720
RUN_LLM = True
MODEL_BACKEND = "q4_gguf"
MODEL_ID = "google/gemma-4-31B-it"
MODEL_PATH_OVERRIDE = ""
Q4_MODEL_ID = "google/gemma-4-31b-it-qat-q4_0-gguf"
Q4_MODEL_PATH_OVERRIDE = "/kaggle/input/models/google/gemma-4/gguf/gemma-4-31b-it-qat-q4_0-gguf/2/gemma-4-31B_q4_0-it.gguf"
Q4_N_CTX, Q4_N_BATCH, Q4_N_UBATCH = 4096, 384, 128
Q4_CONTEXT_CHAR_FALLBACKS = (6000, 3600, 2000, 1000, 400, 0)
MAX_LENGTH, BATCH_ROWS, CHECKPOINT_EVERY = 2048, 2, 10
MAX_REFERENCE_ANSWERS = 12
MAX_DELIB_TOKENS = 12
DELIB_BATCH_ROWS = 2
MAX_DELIB_SAMPLE_ROWS = MAX_DELIB_TEST_ROWS = 0
RUN_DELIBERATION = False
ALLOW_ONLINE_MODEL_FALLBACK = False
MIN_SEMANTIC_REFERENCE_SAMPLE_AGREEMENTS = 0

random.seed(SEED); np.random.seed(SEED)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)
if not INPUT_ROOT.is_dir():
    raise RuntimeError("This production notebook must run inside Kaggle")

test_path = INPUT_ROOT / "competitions/bengali-hallucination/test set.csv"
sample_path = INPUT_ROOT / "competitions/bengali-hallucination/sample submission.csv"
if not test_path.is_file() or not sample_path.is_file():
    raise FileNotFoundError("Competition test/template files are not attached")
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_path)
required = {"id", "context", "prompt_bn", "response_bn"}
if not required.issubset(test.columns):
    raise ValueError(f"test schema missing: {sorted(required-set(test.columns))}")
for name in ("context", "prompt_bn", "response_bn"):
    test[name] = test[name].fillna("").astype(str)
test["id"] = test["id"].astype(str)
sample_submission["id"] = sample_submission["id"].astype(str)
if test.id.duplicated().any() or test.id.tolist() != sample_submission.id.tolist():
    raise ValueError("competition id/order contract failed")

NULL_CONTEXTS = {"", "none", "null", "nan", "n/a", "na", "[null]", "[none]", "[nan]", "[n/a]", "[na]", "<null>", "<none>", "<nan>"}
def has_context(value: object) -> bool:
    folded=str(value or "").strip().casefold()
    if folded in NULL_CONTEXTS or re.fullmatch(r"\[\s*(?:null|none|nan|n/a|na)?\s*\]",folded):
        return False
    return True
def clean_markup(value: object) -> str:
    return str(value or "").replace("\r\n", "\n").replace("\r", "\n").strip()
def row_references(row):
    return (), ()

context_count=int(test.context.map(has_context).sum())
if len(test)==2516 and context_count!=1361:
    raise RuntimeError(f"Phase1 lane canary failed: expected context=1361 closed=1155; got context={context_count} closed={len(test)-context_count}")
print("MORICHIKA test rows:", len(test), "context:", context_count, "closed:", len(test)-context_count)


MORICHIKA test rows: 2516 context: 1361 closed: 1155
